# 合表：订单宽表

以订单商品明细为主表，关联订单、客户、卖家、商品、支付、评价和经纬度，落成 dws_order_wide。

In [1]:
import pandas as pd
from db import query, execute

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

建表时注意支付金额是订单级字段，先聚合到订单再关联，否则一单多笔支付会把金额放大。

In [2]:
ddl = """
DROP TABLE IF EXISTS dws_order_wide;
CREATE TABLE dws_order_wide AS
SELECT
    i.order_id, i.order_item_id, i.product_id, i.seller_id,
    o.customer_id, c.customer_unique_id, c.customer_state, c.zip AS customer_zip,
    s.seller_state, p.category, p.product_weight_g,
    o.order_status, o.purchase_ts, o.delivered_ts, o.estimated_ts,
    i.price, i.freight_value,
    pay.payment_value, pay.payment_cnt, pay.max_installments, pay.main_payment_type,
    r.review_score,
    cg.lat AS c_lat, cg.lng AS c_lng, sg.lat AS s_lat, sg.lng AS s_lng
FROM dwd_items i
JOIN dwd_orders o ON o.order_id = i.order_id
LEFT JOIN dwd_customers c ON c.customer_id = o.customer_id
LEFT JOIN dwd_sellers s ON s.seller_id = i.seller_id
LEFT JOIN dwd_products p ON p.product_id = i.product_id
LEFT JOIN dwd_order_payment pay ON pay.order_id = i.order_id
LEFT JOIN dwd_reviews r ON r.order_id = i.order_id
LEFT JOIN dwd_geo cg ON cg.zip = c.zip
LEFT JOIN dwd_geo sg ON sg.zip = s.zip
"""
execute(ddl)

补上距离和时效几个派生字段：

- `distance_km`：客户与卖家邮编坐标的球面距离
- `delivered_days`：下单到签收
- `promise_days`：下单到承诺送达
- `slack_days`：承诺送达减实际送达，正数表示提前

In [3]:
execute("""
ALTER TABLE dws_order_wide
    ADD COLUMN distance_km DOUBLE,
    ADD COLUMN delivered_days DOUBLE,
    ADD COLUMN promise_days DOUBLE,
    ADD COLUMN slack_days DOUBLE
""")

execute("""
UPDATE dws_order_wide SET
    distance_km = 6371 * ACOS(LEAST(1,
        COS(RADIANS(c_lat)) * COS(RADIANS(s_lat)) * COS(RADIANS(s_lng) - RADIANS(c_lng))
        + SIN(RADIANS(c_lat)) * SIN(RADIANS(s_lat)))),
    delivered_days = TIMESTAMPDIFF(SECOND, purchase_ts, delivered_ts) / 86400.0,
    promise_days = TIMESTAMPDIFF(SECOND, purchase_ts, estimated_ts) / 86400.0,
    slack_days = TIMESTAMPDIFF(SECOND, delivered_ts, estimated_ts) / 86400.0
""")

## 核对

宽表行数应该等于 order_items 的行数；按订单去重后的支付金额合计应该和原始支付表一致。

In [4]:
query("""
SELECT COUNT(*) AS rows_cnt, COUNT(DISTINCT order_id) AS orders
FROM dws_order_wide
""")

,rows_cnt,orders
0,112650,98666


In [5]:
query("""
SELECT ROUND(SUM(payment_value), 2) AS from_wide
FROM (SELECT order_id, MAX(payment_value) AS payment_value FROM dws_order_wide GROUP BY order_id) t
""")

,from_wide
0,15846280.17


In [6]:
query("""
SELECT ROUND(SUM(payment_value), 2) AS from_raw
FROM olist_order_payments_dataset
""")

,from_raw
0,16008872.12


In [7]:
query("""
SELECT order_id, order_status, category, price, payment_value,
       ROUND(distance_km, 1) AS distance_km,
       ROUND(delivered_days, 1) AS delivered_days,
       ROUND(slack_days, 1) AS slack_days
FROM dws_order_wide
WHERE order_status = 'delivered'
LIMIT 5
""")

,order_id,order_status,category,price,payment_value,distance_km,delivered_days,slack_days
0,061aa244b31dd240e079ad7a7ce3f32f,delivered,esporte_lazer,139.80,151.98,8.3,4.1,12.2
1,034fcc4584bd6b6ca0c0718acf969c79,delivered,moveis_decoracao,52.99,64.75,26.8,4.7,16.5
2,0176a6846bcb3b0d3aa3116a9a768597,delivered,esporte_lazer,45.90,53.68,27.7,4.4,16.1
3,04f2e760bb583d5d2f8b88e247d58d09,delivered,moveis_decoracao,37.90,50.98,14.0,2.3,12.3
4,016e4acfc34b9f2abc69795f47c92f0e,delivered,telefonia,45.00,52.78,31.8,3.3,8.1
